# Market Data

## Problem Definition

**Question.** Does the local AAPL 2025 tick Parquet provide a complete, ordered market-data boundary for the research workflow?

**Role in the workflow.** Establish the immutable observed market source; this notebook does not fetch or replace data.

**Inputs.** `data/research_data/market/data/aapl_2025-01-01_2025-12-31.parquet`.

**Outputs.** A validated in-memory tick table and a compact data-quality summary; the source Parquet is unchanged.

**Why this method.** Reading the already acquired Parquet makes the run reproducible without credentials or network access.

**Assumptions.** Timestamps are UTC, prices and sizes are positive, and the file represents only AAPL observations from 2025.

**Handoff.** The local market-data path to `market_structured_bars.ipynb`.


## Real Data Check

The model holdout is not inspected at this raw-data stage. The chronological split is created once event labels exist; until then, checks are limited to schema, ordering, coverage, and missing values.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parents[1]
market_path = PROJECT_ROOT / "data/research_data/market/data/aapl_2025-01-01_2025-12-31.parquet"

market_data = pd.read_parquet(market_path)
required_columns = ["timestamp", "symbol", "price", "size"]
assert market_data.columns.tolist() == required_columns

market_data["timestamp"] = pd.to_datetime(market_data["timestamp"], utc=True)
market_data = market_data.sort_values("timestamp", ignore_index=True)

assert market_data["symbol"].eq("AAPL").all()
assert market_data["timestamp"].dt.year.eq(2025).all()
assert market_data["price"].gt(0).all()
assert market_data["size"].gt(0).all()

quality = pd.Series(
    {
        "rows": len(market_data),
        "start_utc": market_data["timestamp"].min(),
        "end_utc": market_data["timestamp"].max(),
        "duplicate_rows": int(market_data.duplicated().sum()),
        "missing_values": int(market_data.isna().sum().sum()),
        "median_price": float(market_data["price"].median()),
        "median_size": float(market_data["size"].median()),
    },
    name="value",
)
display(quality.to_frame())
display(market_data.head())


## Results, Limitations, and Handoff

The file contains observed trades rather than a regular time grid, so later notebooks use information-driven bars. Duplicate trade rows are reported but not silently removed here because they may represent distinct executions.

The next notebook receives the validated local tick Parquet path. No conclusion in this notebook is evidence of live-trading profitability.
